In [ ]:
import pandas as pd
import numpy as np
import zipfile
import requests

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import itertools, re

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sheikhmuneebahmed115","key":"a1b8cd489ade5e13074a7990c608dfab"}'}

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/

In [ ]:
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d sheikhmuneebahmed115/imbd-data

Dataset URL: https://www.kaggle.com/datasets/sheikhmuneebahmed115/imbd-data
License(s): unknown
100% 67.9M/67.9M [00:00<00:00, 83.0MB/s]



In [ ]:
import zipfile

with zipfile.ZipFile("imbd-data.zip","r") as zip_ref:
    zip_ref.extractall("data")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import itertools, re
import gc

print("=" * 60)
print("LOADING DATA...")
print("=" * 60)


df =  pd.read_csv("data/data.csv")
df.head(20)

LOADING DATA...


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,titleType
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2804960,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",movie
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2509529,"Matthew McConaughey, Anne Hathaway, Michael Ca...",movie
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3155324,"Christian Bale, Heath Ledger, Aaron Eckhart, M...",movie
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1493112,"Sam Worthington, Zoe Saldaña, Sigourney Weaver...",movie
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1553660,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ...",movie
5,293660,Deadpool,7.606,28894,Released,2016-02-09,783100000,108,False,/en971MEXui9diirXlogOrPKmsEn.jpg,...,"20th Century Fox, The Donners' Company, Genre ...",United States of America,English,"superhero, anti hero, mercenary, based on comi...",Tim Miller,"Rhett Reese, Paul Wernick",8.0,1249454,"Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J...",movie
6,299536,Avengers: Infinity War,8.255,27713,Released,2018-04-25,2052415039,149,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,...,Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, s...","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee,...",8.4,1364750,"Robert Downey Jr., Chris Evans, Chris Hemswort...",movie
7,550,Fight Club,8.438,27238,Released,1999-10-15,100853753,139,False,/hZkgoQYus5vegHoetLkCJzb17zJ.jpg,...,"Regency Enterprises, Fox 2000 Pictures, Taurus...",United States of America,English,"dual identity, rage and hate, based on novel o...",David Fincher,"Chuck Palahniuk, Jim Uhls",8.8,2593969,"Edward Norton, Brad Pitt, Helena Bonham Carter...",movie
8,118340,Guardians of the Galaxy,7.906,26638,Released,2014-07-30,772776600,121,False,/uLtVbjvS1O7gXL8lUOwsFOH4man.jpg,...,Marvel Studios,United States of America,English,"spacecraft, based on comic, space, orphan, adv...",James Gunn,"James Gunn, Nicole Perlman, Dan Abnett, Andy L...",8.0,1359535,"Chris Pratt, Zoe Saldaña, Dave Bautista, Vin D...",movie
9,680,Pulp Fiction,8.488,25893,Released,1994-09-10,213900000,154,False,/suaEOtk1N1sgg2MTM7oZd2cfVp3.jpg,...,"Miramax, A Band Apart, Jersey Films",United States of America,"English, Spanish, French","drug dealer, boxer, massage, stolen money, bri...",Quentin Tarantino,"Quentin Tarantino, Roger Avary",8.8,2426800,"John Travolta, Samuel L. Jackson, Uma Thurman,...",movie


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243146 entries, 0 to 243145
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    243146 non-null  int64  
 1   title                 243146 non-null  object 
 2   vote_average          243146 non-null  float64
 3   vote_count            243146 non-null  int64  
 4   status                243146 non-null  object 
 5   release_date          226691 non-null  object 
 6   revenue               243146 non-null  int64  
 7   runtime               243146 non-null  int64  
 8   adult                 243146 non-null  bool   
 9   backdrop_path         98993 non-null   object 
 10  budget                243146 non-null  int64  
 11  homepage              40006 non-null   object 
 12  tconst                243146 non-null  object 
 13  original_language     243146 non-null  object 
 14  original_title        243146 non-null  object 
 15  

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.duplicated(subset=["tconst", "titleType"]).sum()

np.int64(7431)

In [ ]:
df["tconst"] = df["tconst"].str.replace(r"^tt", "", regex=True)

In [ ]:
print(df["tconst"][:10])

0    1375666
1    0816692
2    0468569
3    0499549
4    0848228
5    1431045
6    4154756
7    0137523
8    2015381
9    0110912
Name: tconst, dtype: object


In [ ]:
# df["tconst"] = df["tconst"].astype("int32")

In [ ]:
df[df.duplicated(subset=["tconst"], keep=False)].sort_values("tconst")

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,titleType
188898,285971,The Vault of Mystery,0.0,0,Released,1917-01-02,0,20,False,NaN,...,NaN,NaN,NaN,woman director,"Grace Cunard, Francis Ford","Grace Cunard, Francis Ford",7.1,46,"Francis Ford, Grace Cunard, Jean Hathaway, Pet...",movie
188899,285971,The Vault of Mystery,0.0,0,Released,1917-01-02,0,20,False,NaN,...,NaN,NaN,NaN,woman director,"Grace Cunard, Francis Ford","Grace Cunard, Francis Ford",7.1,46,"Francis Ford, Grace Cunard, Jean Hathaway",movie
157149,1197104,The Purple Mask,0.0,0,Released,1916-12-25,0,330,False,NaN,...,Universal Film Manufacturing Company,United States of America,NaN,NaN,"Grace Cunard, Francis Ford","Grace Cunard, Francis Ford",7.1,46,"Francis Ford, Grace Cunard, Jean Hathaway, Pet...",movie
157150,1197104,The Purple Mask,0.0,0,Released,1916-12-25,0,330,False,NaN,...,Universal Film Manufacturing Company,United States of America,NaN,NaN,"Grace Cunard, Francis Ford","Grace Cunard, Francis Ford",7.1,46,"Francis Ford, Grace Cunard, Jean Hathaway",movie
164786,1377793,Muratti privat,0.0,0,Released,1935-12-01,0,3,False,NaN,...,Fischinger Studio,NaN,NaN,NaN,Oskar Fischinger,NaN,6.3,65,NaN,short
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166288,1512853,Sisters of House Black,0.0,0,Released,2019-10-30,0,42,False,NaN,...,NaN,United Kingdom,NaN,NaN,"Thomas Fisher, Petros L. Ioannou",Kelsey Ellison,6.8,164,NaN,short
227214,1033156,Sisters of House Black,7.0,0,Released,2019-10-23,0,42,False,NaN,...,Ellivision,United Kingdom,English,NaN,"Thomas Fisher, Petros L. Ioannou",Kelsey Ellison,6.8,164,"Kelsey Ellison, Hannah Snow",short
227215,1033156,Sisters of House Black,7.0,0,Released,2019-10-23,0,42,False,NaN,...,Ellivision,United Kingdom,English,NaN,"Thomas Fisher, Petros L. Ioannou",Kelsey Ellison,6.8,164,NaN,short
164144,1431223,In This Cold Place,0.0,0,Released,2017-06-19,0,3,False,NaN,...,Little Idiot,"United Kingdom, United States of America",English,NaN,Steve Cutts,Moby,7.0,81,Moby,short


In [ ]:
df["title"] = df["title"].str.strip()

In [ ]:
df["merge_key"] = df["tconst"].astype(str) + "_" + df["title"]

In [ ]:
def first_valid(series):
    valid = series.dropna()
    return valid.iloc[0] if len(valid) > 0 else None



In [ ]:
df_clean = df.groupby("merge_key").agg({

    # identity
    "tconst": "first",
    "title": "first",
    "id": "first",

    # ratings (numerical blending)
    "vote_average": "mean",
    "averageRating": "mean",

    # counts (add them up)
    "vote_count": "sum",
    "numVotes": "sum",

    # financials
    "revenue": "max",     # avoids duplicate inflation
    "budget": "max",

    # time
    "runtime": "mean",

    # important metadata (fill from first available)
    "release_date": first_valid,
    "genres": first_valid,
    "directors": first_valid,
    "writers": first_valid,
    "cast": first_valid,
    "production_companies": first_valid,
    "spoken_languages": first_valid,
    "keywords": first_valid,

    # misc
    "popularity": "mean"

}).reset_index(drop=True)

In [ ]:
df_clean.duplicated(subset=["tconst"]).sum()

np.int64(360)

In [ ]:
df_clean[df_clean.duplicated(subset=["tconst"], keep=False)] \
.sort_values("tconst")

,tconst,title,id,vote_average,averageRating,vote_count,numVotes,revenue,budget,runtime,release_date,genres,directors,writers,cast,production_companies,spoken_languages,keywords,popularity
1503,0008476,The Purple Mask,1197104,0.0,7.1,0,92,0,0,330.0,1916-12-25,Action,"Grace Cunard, Francis Ford","Grace Cunard, Francis Ford","Francis Ford, Grace Cunard, Jean Hathaway, Pet...",Universal Film Manufacturing Company,None,None,1.3410
1504,0008476,The Vault of Mystery,285971,0.0,7.1,0,92,0,0,20.0,1917-01-02,None,"Grace Cunard, Francis Ford","Grace Cunard, Francis Ford","Francis Ford, Grace Cunard, Jean Hathaway, Pet...",None,None,woman director,1.0980
8274,0026880,Popular Science J-5-1,1606897,0.0,5.0,0,18,0,0,11.0,1935-05-31,Documentary,Robert Carlisle,None,Gayne Whitman,"Fairbanks-Carlisle Productions, Paramount Pict...",English,"dam, educational, one-reeler, boulder dam, con...",0.0071
8275,0026880,Popular Science J6-2,233695,0.0,5.0,0,18,0,0,11.0,1935-09-20,None,Robert Carlisle,None,Gayne Whitman,None,None,None,0.6000
26631,0066904,Le Chagrin et la Pitié,1418448,0.0,8.1,0,8954,0,0,251.0,None,None,Marcel Ophüls,"André Harris, Marcel Ophüls","Helmut Tausend, Marcel Verdier, Alexis Grave, ...",None,None,None,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233654,9253284,Star Wars: Andor,1326341,0.0,8.6,0,272825,0,0,582.0,None,"Science Fiction, Fantasy, Action, Adventure, D...","Toby Haynes, Ariel Kleiman, Benjamin Caron, Su...","Tony Gilroy, George Lucas, Dan Gilroy, Beau Wi...",None,Walt Disney Studios,None,None,0.0000
234072,9357776,Wilford 'MOTHERLOVING' Warfstache,1635291,0.0,9.0,0,331,0,42000,19.0,2018-08-06,"Comedy, Drama",Mark Fischbach,Mark Fischbach,"Mark Fischbach, Mick Lauer",None,English,markiplier,0.0357
234073,9357776,Wilford 'Motherloving' Warfstache,1059449,10.0,9.0,1,331,0,0,19.0,2018-08-05,"Comedy, Crime, Drama",Mark Fischbach,Mark Fischbach,"Mark Fischbach, Mick Lauer",Markiplier,None,"mustache, crazy, pink",0.6000
235675,9810488,The Appendices Part 1: From Book to Vision,1654600,0.0,8.8,0,92,0,0,0.0,2002-11-12,None,None,None,None,None,None,None,0.0000


In [ ]:
df["data_score"] = df.notna().sum(axis=1)

In [ ]:
df = df.sort_values(["tconst", "data_score"], ascending=[True, False])

In [ ]:
df_clean = df.drop_duplicates(subset=["tconst"], keep="first")

In [ ]:
df_clean[df_clean.duplicated(subset=["tconst"], keep=False)] \
.sort_values("tconst")

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,titleType,merge_key,data_score


In [ ]:
df_clean.to_csv("advanced-imdb.csv", index=False)


In [ ]:
!mkdir advanced-imdb
!mv advanced-imdb.csv advanced-imdb/


In [ ]:

# ====== 3. write kaggle metadata ======
metadata = {
    "title": "Advanced IMDb Dataset",
    "id": "sheikhmuneebahmed115/advanced-imdb",
    "licenses": [{"name": "CC0-1.0"}]
}

import json
with open("advanced-imdb/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

# ====== 4. install kaggle api ======
!pip install -q kaggle

# ====== 5. upload kaggle.json (you must upload manually in colab) ======
from google.colab import files
files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ====== 6. create kaggle dataset ======
!kaggle datasets create -p advanced-imdb

Saving kaggle.json to kaggle.json
Dataset creation error: The requested title "Advanced IMDb Dataset" is already in use by a dataset. Please choose another title.


In [ ]:
!kaggle datasets download sheikhmuneebahmed115/imbd-data

Dataset URL: https://www.kaggle.com/datasets/sheikhmuneebahmed115/imbd-data
License(s): unknown
100% 67.9M/67.9M [00:00<00:00, 83.0MB/s]



In [ ]:
!kaggle datasets download -d sheikhmuneebahmed115/advanced-imdb

Dataset URL: https://www.kaggle.com/datasets/sheikhmuneebahmed115/advanced-imdb
License(s): CC0-1.0
100% 3.58M/3.58M [00:00<00:00, 109MB/s]



In [ ]:
# import zipfile

# with zipfile.ZipFile("advanced-imdb.zip","r") as zip_ref:
#     zip_ref.extractall("advanced-imbd")


In [ ]:
# import json

# metadata = {
#     "title": "Advanced IMDb Dataset",
#     "id": "sheikhmuneebahmed115/advanced-imdb",
#     "licenses": [
#         {
#             "name": "CC0-1.0"
#         }
#     ]
# }

# with open("advanced-imdb/dataset-metadata.json", "w") as f:
#     json.dump(metadata, f, indent=4)

In [ ]:
# !kaggle datasets version -p advanced-imdb -m "upload train + zip dataset + ratings"

Starting upload for file advanced-imdb-train.csv
100% 7.87M/7.87M [00:00<00:00, 10.7MB/s]
Upload successful: advanced-imdb-train.csv (8MB)
Starting upload for file advanced-imdb.csv
100% 155M/155M [00:02<00:00, 67.6MB/s]
Upload successful: advanced-imdb.csv (155MB)
Skipping folder: .ipynb_checkpoints; use '--dir-mode' to upload folders
Starting upload for file ratings.csv
100% 2.37M/2.37M [00:00<00:00, 3.17MB/s]
Upload successful: ratings.csv (2MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/sheikhmuneebahmed115/advanced-imdb
